In [1]:
#import require library
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset,DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

In [2]:
#load data set
df = pd.read_csv('fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [12]:
#separed the x data input dat
x=df.iloc[:,1:].values
y = df.iloc[:,0].values

In [13]:
x.shape

(6000, 784)

In [14]:
y.shape

(6000,)

In [15]:
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

2.11.0+cu128
12.8
True


In [16]:
Device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Device

device(type='cuda')

In [17]:
#split dataset into train and test part
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=43)

In [18]:
#normalization the images 
x_train = x_train/255.0
x_test = x_test/255.0

In [19]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features,dtype=torch.float32)
        self.labels = torch.tensor(labels,dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [20]:
train_data = CustomDataset(x_train,y_train)
test_data = CustomDataset(x_test,y_test)

In [21]:
len(train_data)

4800

In [22]:
len(test_data)

1200

In [42]:
class MyAnn(nn.Module):
    def __init__(self, number_dim,output_dim,learning_rate,drop_rate,hidden_layear,neurons_par_layear):
        super().__init__()

        layers =[]

        for i in range(hidden_layear):
            
                layers.append(nn.Linear(number_dim,neurons_par_layear))
                layers.append(nn.BatchNorm1d(neurons_par_layear))
                layers.append(nn.ReLU())
                layers.append(nn.Dropout(p=drop_rate))

                number_dim = neurons_par_layear
        layers.append(nn.Linear(neurons_par_layear,output_dim))

        self.model = nn.Sequential(*layers)

    def forward(self,x):
         
        return self.model(x)
            

In [43]:
##define Objective function
def objective(trial):
  # next hyperparameter values from the search space
  hidden_layear = trial.suggest_int('hidden_layear',1,5)
  neurons_par_layear = trial.suggest_int("neurons_par_layear",8,128)
  drop_rate = trial.suggest_float("drop_rate",0.1,0.5)
  batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
  epochs = trial.suggest_int("epochs", 10, 50, step=10)

  #model inti
  number_dim =784
  output_dim =10
  
  train_loader = DataLoader(train_data,batch_size=batch_size,shuffle=True,pin_memory=True)
  test_loader = DataLoader(test_data,batch_size=batch_size,shuffle=False,pin_memory=True)

  #create model
  Model = MyAnn(number_dim,output_dim,learning_rate,drop_rate,hidden_layear,neurons_par_layear)
  model = Model.to(Device)

  #create loss function 
  loss_function = nn.CrossEntropyLoss()

  #create optimizer
  optimizer = optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)
  if optimizer_name == "Adam":
    optimizer=optim.Adam(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
  if optimizer_name =="SGD":
    optimizer=optim.SGD(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
  else:
    optimizer=optim.RMSprop(model.parameters(),lr=learning_rate,weight_decay=weight_decay)

  for epoch in range(epochs):
    for batch_features,batch_labels in train_loader:
    
      #batch conver to devices
      batch_features,batch_labels = batch_features.to(Device),batch_labels.to(Device)
      #forward
      outputs = model(batch_features)
      #calculate loss
      loss = loss_function(outputs,batch_labels)
      #backforward
      optimizer.zero_grad()
      loss.backward()

      optimizer.step()
  #evaluate the model
  model.eval()
  # evaluation on test data
  total = 0
  correct = 0

  with torch.no_grad():

    for batch_features, batch_labels in test_loader:

      # move data to gpu
      batch_features, batch_labels = batch_features.to(Device), batch_labels.to(Device)

      outputs = model(batch_features)

      _, predicted = torch.max(outputs, 1)

      total = total + batch_labels.shape[0]

      correct = correct + (predicted == batch_labels).sum().item()

      accuracy = correct/total

      return accuracy
    
  

In [44]:
!pip install optuna

In [45]:
import optuna
study = optuna.create_study(direction='maximize')

[I 2026-06-19 16:07:58,568] A new study created in memory with name: no-name-ba16c064-7a5e-4c43-ab7e-1bb7e4779b22


In [46]:
study.optimize(objective,n_trials=10)

[I 2026-06-19 16:08:24,754] Trial 0 finished with value: 0.9375 and parameters: {'hidden_layear': 2, 'neurons_par_layear': 122, 'drop_rate': 0.2918023402281878, 'batch_size': 64, 'optimizer': 'RMSprop', 'weight_decay': 0.0007511403245988109, 'learning_rate': 0.0012506805569142616, 'epochs': 40}. Best is trial 0 with value: 0.9375.
[I 2026-06-19 16:09:33,197] Trial 1 finished with value: 0.25 and parameters: {'hidden_layear': 3, 'neurons_par_layear': 19, 'drop_rate': 0.363263507855084, 'batch_size': 16, 'optimizer': 'SGD', 'weight_decay': 9.520072219807834e-05, 'learning_rate': 4.353931899886143e-05, 'epochs': 40}. Best is trial 0 with value: 0.9375.
[I 2026-06-19 16:11:14,906] Trial 2 finished with value: 0.9375 and parameters: {'hidden_layear': 4, 'neurons_par_layear': 101, 'drop_rate': 0.3141654011272177, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 0.0008647004554851168, 'learning_rate': 0.001756687187622797, 'epochs': 50}. Best is trial 0 with value: 0.9375.
[I 2026-06

In [47]:
study.best_value

1.0

In [48]:
study.best_params

{'hidden_layear': 3,
 'neurons_par_layear': 61,
 'drop_rate': 0.2754362757472647,
 'batch_size': 16,
 'optimizer': 'RMSprop',
 'weight_decay': 0.0003314631192606085,
 'learning_rate': 0.000299294381307655,
 'epochs': 50}